# Packages import

In [19]:
import os
import re
import yaml
import requests
import pandas as pd
from requests.auth import HTTPBasicAuth
from bs4 import BeautifulSoup

# Apollo scraper

In [ ]:
group_id = input("Enter group ID: ")
url = f"https://planzajec.uek.krakow.pl/index.php?typ=G&id={group_id}&okres=1"

In [ ]:
with open("config.yaml", "r", encoding="utf-8") as yf:
    config = yaml.load(yf, Loader=yaml.SafeLoader)
username = config.get("credentials", {}).get("username") if config else None
password = config.get("credentials", {}).get("password") if config else None
auth = HTTPBasicAuth(username, password)

In [ ]:
response = requests.get(url, auth=auth)
response.encoding = "UTF-8"
print(response.status_code)

In [ ]:
page_dom = BeautifulSoup(response.text, "html.parser")

In [ ]:
group = page_dom.select_one("div.grupa").get_text()
print(group)

In [ ]:
classes_tag = page_dom.select_one("table")
with open("temp.html", "w", encoding="UTF-8") as hf:
    hf.write(str(classes_tag))

In [ ]:
classes = pd.read_html("temp.html", encoding="UTF-8")[0]
os.remove("temp.html")

In [ ]:
classes = classes.loc[classes["Typ"].isin(["ćwiczenia", "wykład", "egzamin"])]

In [ ]:
classes[["Day", "Start time", "hyphen", "End time", "Duration"]] = classes["Dzień, godzina"].str.split(" ", expand=True)

In [ ]:
classes["Sala"] = classes["Sala"].str.replace(
    r"(lab\.).*",
    r"\1",
    regex=True
)

In [ ]:
if not os.path.exists("./schedules"):
    os.mkdir("./schedules")

In [ ]:
classes.to_csv(f"schedules/{group}.csv", encoding="UTF-8")

In [ ]:
print(classes)